# Optimization for Data Science - Linear Programming

Andrés F. López-Lopera <br/>
IMAG, Université de Montpellier

_____

In this notebook, you will learn how to use different Python toolboxes to tackle linear programming tasks.

<div class="alert alert-danger">
Caution: Execute the cells sequentially!
<div>

## 1. Linear programming using Scipy

**Scipy** is a **NumPy**-based Python toolbox containing fundamental algorithms for scientific computing ([https://scipy.org/](https://scipy.org/)), including algorithms for **optimization**, integration, interpolation, eigenvalue problems, algebraic equations, differential equations, statistics, and more.

**Advantages:**
- No need for external solvers, pure Python + NumPy.
- Uses methods like **simplex**, interior point, and HiGHS.
- Good for small to medium-sized LP problems.
- Has an interface to a high-performance linear programming solver
 
**Disadvantages:**
- Needs objective function and constraints to be in a given form.
- Cannot solve integer programming problems.

Scipy can be installed via `pip install`. The optimization methods, including the ones for linear programming (`linprog`), can be called from `scipy.optimize` utility. 

In [1]:
%pip install SciPy

  Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached numpy-2.5.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached scipy-1.18.1-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached numpy-2.5.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [SciPy]32m1/2 [SciPy]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from scipy.optimize import linprog

Before starting to code, it is good practice to read the help documentation of the functions you plan to use, as this clarifies their proper usage and provides details about their input and output variables.

In [3]:
?linprog

Signature:
linprog(
    c,
    A_ub=None,
    b_ub=None,
    A_eq=None,
    b_eq=None,
    bounds=(0, None),
    method='highs',
    callback=None,
    options=None,
    x0=None,
    integrality=None,
)
Docstring:
Linear programming: minimize a linear objective function subject to linear
equality and inequality constraints.

Linear programming solves problems of the following form:

.. math::

    \min_x \ & c^T x \\
    \mbox{such that} \ & A_{ub} x \leq b_{ub},\\
    & A_{eq} x = b_{eq},\\
    & l \leq x \leq u ,

where :math:`x` is a vector of decision variables; :math:`c`,
:math:`b_{ub}`, :math:`b_{eq}`, :math:`l`, and :math:`u` are vectors; and
:math:`A_{ub}` and :math:`A_{eq}` are matrices.

Alternatively, that's:

- minimize ::

    c @ x

- such that ::

    A_ub @ x <= b_ub
    A_eq @ x == b_eq
    lb <= x <= ub

Note that by default ``lb = 0`` and ``ub = None``. Other bounds can be
specified with ``bounds``.

Parameters
----------
c : 1-D array
    The coefficients of the lin

**Question.** What can you observe after reading the documentation?

**Exercise 1.** By following the documentation, write a code allowing to solve the linear program $(\mathcal{P})$ given by

$$
(\mathcal{P}) \qquad
\begin{aligned}
    \begin{array}{cccc}
        \text{max}  & 5x_1 &+9x_2 & \\
    	\text{s.c.} & x_1 &+x_2 & \le 4 \\
        			& 2x_1 &+x_2 & \le 6\\
					& x_1, & x_2 & \ge 0
    \end{array}
\end{aligned}
$$

Recall that $(\mathcal{P})$ can be expressed in matrix form
$$
(\mathcal{P}) \qquad
\begin{aligned}
    \begin{array}{ccc}
        \text{max}  & c^\top x & \\
    	\text{s.c.} & A x & \le b \\
					& x & \ge 0
    \end{array},
\end{aligned}
$$
with $x = (x_1, x_2)^\top$, $c = (5, 9)^\top$, $b = (4, 6)^\top$ and $A = \begin{pmatrix} 1 & 1 \\ 2 & 1 \end{pmatrix}$.

Use the **simplex algorithm** as the optimizer method.

In [ ]:
# writing the linear program (LP) as numpy-based objects
import numpy as np
## defining the objective function to be minimized
c = np.array([-5, -9]) #we inverse the sign of the coefficients to convert the maximization problem into a minimization one

## defining the constraints
A = np.array([[1, 1],
            [2, 1]])
b = np.array([4, 6])
bounds = [(0, None), (0, None)] 

# solving the LP via the linprog function from scipy
res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='simplex')

# checking the results
print("Status:", res.status)
print("Message:", res.message)
print("Nb iterations:", res.nit)

# displaying the results
print("Results after optimization")
print("x* = ", res.x, "\ny(x*) =", -res.fun)

Status: 0
Message: Optimization terminated successfully.
Nb iterations: 4
Results after optimization
x* =  [0. 4.] 
y(x*) = 36.0


/tmp/ipykernel_27238/628058178.py:13: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='simplex')


**Exercise 2.** Solve the following linear programs

$$
(\mathcal{P}_1) \qquad
\begin{aligned}
    \begin{array}{cccc}
        \text{max}  & 4x_1 &+5x_2 & \\
    	\text{s.c.} & &+x_2 & \le 2 \\
        			& x_1 &+x_2 & \le 3\\
                    & 2x_1 &+x_2 & \le 5\\                    
					& x_1, & x_2 & \ge 0
    \end{array}
\end{aligned}
\hspace{5cm}
(\mathcal{P}_2) \qquad
\begin{aligned}
    \begin{array}{cccc}
        \text{max}  & x_1 &+2x_2 & \\
    	\text{s.c.} & x_1 & & \le 5 \\
        			& x_1 &+x_2 & \ge 3\\
					& x_1, & x_2 & \ge 0
    \end{array}
\end{aligned}
$$




In [ ]:
# writing the linear program (P1) as numpy-based objects

## defining the objective function to be minimized
c = np.array([-4, -5])

## defining the constraints
A = np.array([[0, 1],
              [1, 1],
              [2, 1]])
b = np.array([2, 3, 5])
bounds = [(0, None), (0, None)]

# solving the LP via the linprog function from scipy
res = linprog(c, A_ub = A, b_ub = b, bounds = bounds, method = 'simplex')

# checking the results
print("Status:", res.status)
print("Message:", res.message)
print("Nb iterations:", res.nit)

# displaying the results
print("Results after optimization")
print("x* = ", res.x, "\ny(x*) =", -res.fun)

Status: 0
Message: Optimization terminated successfully.
Nb iterations: 3
Results after optimization
x* =  [1. 2.] 
y(x*) = 14.0


/tmp/ipykernel_27238/4005571188.py:12: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(c, A_ub = A, b_ub = b, bounds = bounds, method = 'simplex')


In [11]:
# writing the linear program (P2) as numpy-based objects

## defining the objective function to be minimized
c = np.array([-1, -2])

## defining the constraints
A = np.array([[1, 0], [-1, -1]])
b = np.array([5, -3])
bounds = [(0, None), (0, None)]

# solving the LP via the linprog function from scipy
res = linprog(c, A_ub = A, b_ub = b, bounds = bounds, method = 'simplex')

# checking the results
print("Status:", res.status)
print("Message:", res.message)
print("Nb iterations:", res.nit)

# displaying the results
print("Results after optimization")
print("x* = ", res.x, "\ny(x*) =", -res.fun)

Status: 3
Message: Optimization failed. The problem appears to be unbounded.
Nb iterations: 2
Results after optimization
x* =  [5. 0.] 
y(x*) = 5.0


/tmp/ipykernel_27238/3066158512.py:12: DeprecationWarning: `method='simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = linprog(c, A_ub = A, b_ub = b, bounds = bounds, method = 'simplex')


**Exercise 3.** Solve **Exercise 7** from the guided exercises (TD).


In [ ]:
# writing the linear program as numpy-based objects

## 2. Linear programming using PuLP

**PuLP** is an LP modeler written in Python ([https://pypi.org/project/PuLP/](https://pypi.org/project/PuLP/)). It can generate LP files and call GLPK, COIN CLP/CBC, CPLEX, and GUROBI to solve linear problems.

**Advantages:**
- Beginner-friendly.
- Pure Python library for modeling LP problems.
- Works with many solvers: CBC (default, open-source), GLPK, CPLEX, Gurobi, etc.

**Disadvantages:**
- Cannot model nonlinear optimization problems.
- Some of the solvers need to be installed manually.

Similar to Scipy, PuLP can be installed via `pip install`. The LP optimization method can be called from `pulp.LpProblem` utility. However, some optimizers are not automatically included with PuLP, for instance, the simplex-based one `glpk`. It needs to be installed via the console.
- **Windows:** `conda install -c conda-forge glpk`
- **Linux (Ubuntu/Debian):** `sudo apt-get install -y glpk-utils libglpk-dev`
- **macOS:** `brew install glpk`

In [ ]:
!pip install pulp==3.0.2
!apt-get install -y glpk-utils

In [ ]:
import pulp

?pulp.LpProblem

**Example.** As an example, we consider the LP from **Exercise 1**.

In [ ]:
print(pulp.__version__)

In [ ]:
?pulp.LpVariable

In [ ]:
# defining the LP problem
LP = pulp.LpProblem("(P)", pulp.LpMaximize)

# defining the variables and the bounds
x1 = pulp.LpVariable("x1", lowBound = 0)
x2 = pulp.LpVariable("x2", lowBound = 0)

# defining the objective function
LP += 5*x1 + 9*x2

# defining the constraints
LP += x1 + x2 <= 4
LP += 2*x1 + x2 <= 6

# solving the LP 
status = LP.solve(pulp.GLPK(msg=True))

# displaying the results
print("Results after optimization")
print("Status:", pulp.LpStatus[status])
print("x* = ", [x1.value(), x2.value()], "\ny(x*) =", pulp.value(LP.objective))

**Exercise 4.** Solve the LP problems in **Exercises 2** and **3** using the PuLP toolbox.

In [ ]:
# defining the LP problem (P1)

# defining the variables and the bounds

# defining the objective function

# defining the constraints

# solving the LP

# displaying the results
print("Results after optimization")
print("Status:", pulp.LpStatus[status])
print("x* = ", [x1.value(), x2.value()], "\ny(x*) =", pulp.value(LP.objective))

In [ ]:
# defining the LP problem (P2)

# defining the variables and the bounds

# defining the objective function

# defining the constraints

# solving the LP

# displaying the results
print("Results after optimization")
print("Status:", pulp.LpStatus[status])
print("x* = ", [x1.value(), x2.value()], "\ny(x*) =", pulp.value(LP.objective))

In [ ]:
# defining the LP problem (P3)

# defining the variables and the bounds

# defining the objective function

# defining the constraints

# solving the LP

# displaying the results
print("Results after optimization")
print("Status:", pulp.LpStatus[status])
print("x* = ", [x1.value(), x2.value(), x3.value()], "\ny(x*) =", pulp.value(LP.objective))

## 3. Linear programming using CVXPY (optional)

**CVXPY** is an open source Python-embedded modeling language for convex optimization problems ([https://www.cvxpy.org/](https://www.cvxpy.org/)). It lets you express your problem in a natural way that follows the math, rather than in the restrictive standard form required by solvers.

**Advantages:**
- NumPy-like syntax.
- General Convex Optimization, modern interface
- Supports LP, QP, conic programming, semidefinite programming, etc.
- Solver-agnostic (can call ECOS, SCS, GLPK, Gurobi, CPLEX).

**Disadvantages:**
- Some of the solvers need to be installed manually.

Similar to PuLP, CVXPY requires independent installation for some solvers, including `glpk`.
- **Windows:** `conda install -c conda-forge glpk`
- **Linux (Ubuntu/Debian):** `sudo apt-get install -y glpk-utils libglpk-dev`
- **macOS:** `brew install glpk`

In [ ]:
!pip install cvxpy
!pip install cvxpy[glpk]

In [ ]:
import cvxpy as cp

?cp.Problem

**Exercise 5.** Solve the LP problems in **Exercises 1 - 3** using the cvxpy toolbox.

In [ ]:
# defining the variables and the bounds
x1 = cp.Variable(name='x1', nonneg=True)
x2 = cp.Variable(name='x2', nonneg=True)

# defining the objective function
obj = cp.Maximize(5*x1 + 9*x2)

# defining the constraints
constraints = [x1 + x2 <= 4,
               2*x1 + x2 <= 6,
               x1 >= 0,
               x2 >= 0]

# defining the LP problem
LP = cp.Problem(obj, constraints)

# solving the LP 
LP.solve(solver=cp.GLPK)

# displaying the results
print("Results after optimization")
print("Status:", LP.status)
print("x* = ", [x1.value, x2.value], "\ny(x*) =", LP.value)

In [ ]:
# defining the variables and the bounds

# defining the objective function

# defining the constraints

# defining the LP problem

# solving the LP 

# displaying the results
print("Results after optimization")
print("Status:", LP.status)
print("x* = ", [x1.value, x2.value], "\ny(x*) =", LP.value)

## 4. Linear programming using the geometric algorithm (implementation from scratch)

**Steps of the geometric algorithm**
1. Start from an extreme point $x = (x_1, x_2)$ from the feasible region.
2. Determine an edge along which the objective increases. If none exists, $x$ is optimal (**stop**).
3. Move along the edge to the next extreme point $x$. If none exists, the problem is unbounded (**stop**). Otherwise, set $x \leftarrow x'$ and return to step 2.

**Exercise 6.** Implement the geometric algorithm from scratch. Test your implementation on the LP in **Exercise 1**. 
$$
(\mathcal{P}) \qquad
\begin{aligned}
    \begin{array}{cccc}
        \text{max}  & 5x_1 &+9x_2 & \\
    	\text{s.c.} & x_1 &+x_2 & \le 4 \\
        			& 2x_1 &+x_2 & \le 6\\
					& x_1, & x_2 & \ge 0
    \end{array}
\end{aligned}
$$

In [ ]:
import numpy as np